# The guide, runnable

This notebook mirrors [GUIDE.md](../GUIDE.md) cell for cell: the five steps of a
target-vs-reality simulation, worked through the Mølmer–Sørensen two-qubit gate
(analytic Magnus target vs. a detuned Lamb–Dicke "reality"). Read the guide alongside —
this file is only the code.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # unnecessary after `pip install -e .`

import numpy as np
import matplotlib.pyplot as plt

import htdse as ht
from htdse.submodules.spin import pauli_term
from htdse.submodules.harmonic_oscillator import fock
from htdse.submodules.molmer_sorensen import MSMagnus, ms_lamb_dicke1
from htdse.util import otimes
from htdse.core.plotting import plot_populations

delta, eta = 1.0, 0.1                    # gate detuning, Lamb-Dicke parameter
Omega = delta / (eta * np.sqrt(2))       # pi/4 entangling-angle calibration
T = 2 * np.pi / delta                    # loop-closure time
n_max = 12                               # Fock truncation
b = np.array([1, 1]) / np.sqrt(2)        # COM-mode participation

## Step 1 — compose the target

The analytic Magnus gate: a mechanism defined as `.unitary(t)`, no ODE involved.

In [ ]:
target = MSMagnus(b, eta, delta, Omega, [0.0, 0.0], n_max)
target

## Step 2 — build the realized model

The pre-RWA Lamb–Dicke builder with a 5% detuning miscalibration; error injection is
just composition (same frame, literal addition).

In [ ]:
eps = 0.05
H_real = ms_lamb_dicke1(b, eta, delta * (1 + eps), Omega, [0.0, 0.0], n_max, rwa=True)
H_err = H_real + pauli_term("Z0", coeff=0.02)   # optional extra: stray sigma_z on ion 0
H_real

## Step 3 — evolve with the class matching your object

A ket, so `HamiltonianEvolution`. Nothing integrates until `state_at` is called.

In [ ]:
psi0 = ht.Operator(otimes(ht.ket("00"), fock(0, n_max)))   # |00> x |vac>
with ht.quiet():
    ev = ht.HamiltonianEvolution(H_real, psi0)
    psi_T = ev.state_at(T)
psi_ideal = np.asarray(target.unitary(T)) @ np.asarray(psi0)

## Step 4 — reconcile Hilbert-space mismatches

The realized state lives on spins ⊗ mode. `trace_out` (registry supplied by the term
layer) brings it down to the spins.

In [ ]:
rho_spins = ev.trace_out("mode", t=T)
print("reduced state:", np.asarray(rho_spins).shape)
print("spin populations:", np.real(np.diag(np.asarray(rho_spins))).round(4))

## Step 5 — compare and visualize

In [ ]:
print(f"gate fidelity vs analytic target: {ht.fidelity(psi_ideal, psi_T):.4f}")

ts = np.linspace(0, T, 60)
with ht.quiet():
    F = ht.compare_over(ts,
                        ht.UnitaryEvolution(target),                    # analytic U(t)
                        ht.UnitaryEvolution(H_real, dim=psi0.shape[0]), # solved U(t)
                        metric=ht.process_fidelity)
plt.plot(ts, F)
plt.xlabel("t"); plt.ylabel("process fidelity"); plt.title("target vs realized over the gate")
plt.show()

## Where next

The recipes in [GUIDE.md](../GUIDE.md) cover dissipation, Trotterization, sparse
storage for large Hilbert spaces, and hand-written mechanisms. The remaining demos
(01–05) each exercise one of them in depth.